# MI-Net: Multi-scale Integration Network

**Bio-Inspired Deep Learning Model**

**Bio-Inspiration**: Multi-scale integration inspired by V1 receptive fields  
**Deep Learning Enhancement**: Data-driven pathway aggregation  
**Improvement Area**: Robustness across resolutions

Implements multi-scale feature integration for edge detection.

In [ ]:
# Configuration
from pathlib import Path
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'torchvision', 'opencv-python', 'matplotlib', 'numpy', 'tqdm', 'scikit-learn'], check=False)

import torch, torch.nn as nn, torch.nn.functional as F, numpy as np, cv2
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.metrics import average_precision_score

PROJECT_ROOT = Path('..')
DATASET_ROOT = PROJECT_ROOT / 'datasets' / 'HED_Small'
OUTPUT_DIR = PROJECT_ROOT / 'bio DL' / 'outputs' / 'MI-Net'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## MI-Net Architecture

In [ ]:
class MultiScaleBlock(nn.Module):
    """Multi-scale receptive fields"""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.scale1 = nn.Conv2d(in_ch, out_ch//3, 3, padding=1)
        self.scale2 = nn.Conv2d(in_ch, out_ch//3, 5, padding=2)
        self.scale3 = nn.Conv2d(in_ch, out_ch//3, 7, padding=3)
    def forward(self, x):
        return torch.cat([F.relu(self.scale1(x)), F.relu(self.scale2(x)), F.relu(self.scale3(x))], dim=1)

class MINet(nn.Module):
    def __init__(self):
        super().__init__()
        self.ms1 = MultiScaleBlock(3, 64)
        self.ms2 = MultiScaleBlock(64, 128)
        self.ms3 = MultiScaleBlock(128, 256)
        self.edge1, self.edge2, self.edge3 = nn.Conv2d(64, 1, 1), nn.Conv2d(128, 1, 1), nn.Conv2d(256, 1, 1)
        self.fusion = nn.Conv2d(3, 1, 1)
    def forward(self, x):
        h, w = x.shape[2:]
        e1, e2, e3 = self.ms1(x), self.ms2(F.max_pool2d(self.ms1(x), 2)), self.ms3(F.max_pool2d(self.ms2(F.max_pool2d(self.ms1(x), 2)), 2))
        s1, s2, s3 = self.edge1(e1), self.edge2(e2), self.edge3(e3)
        s1, s2, s3 = [F.interpolate(s, (h, w), mode='bilinear', align_corners=False) for s in [s1, s2, s3]]
        return torch.sigmoid(self.fusion(torch.cat([s1, s2, s3], dim=1))), [torch.sigmoid(s) for s in [s1, s2, s3]]

model = MINet().to(DEVICE).eval()
print(f"✓ MI-Net created ({sum(p.numel() for p in model.parameters()):,} params)")

In [ ]:
# Dataset & Evaluation (compact)
class EdgeDataset(Dataset):
    def __init__(self, root, split='test'):
        self.img_dir, self.gt_dir = root / split / 'images', root / split / 'edges'
        self.images = sorted(list(self.img_dir.glob('*.jpg')) + list(self.img_dir.glob('*.png')))[:20]
    def __len__(self): return len(self.images)
    def __getitem__(self, idx):
        img_path = self.images[idx]
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        gt_path = self.gt_dir / img_path.name.replace('.jpg', '.png')
        gt = cv2.imread(str(gt_path), 0).astype(np.float32) / 255.0 if gt_path.exists() else np.zeros(img.shape[:2], dtype=np.float32)
        return torch.from_numpy(img.transpose(2, 0, 1)), torch.from_numpy(gt), img_path.name

def compute_metrics(preds, labels):
    threshs, all_preds, all_labels, ois = np.linspace(0.05, 0.95, 30), [], [], []
    for pred, label in zip(preds, labels):
        label_tol = cv2.dilate((label > 0.5).astype(np.float32), cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))).flatten()
        pred_smooth = cv2.GaussianBlur(pred, (3,3), 0).flatten()
        all_preds.append(pred_smooth); all_labels.append(label_tol)
        ois.append(max([2*np.sum((pred_smooth>=t)*label_tol)/(2*np.sum((pred_smooth>=t)*label_tol)+np.sum((pred_smooth>=t)*(1-label_tol))+np.sum((pred_smooth<t)*label_tol)+1e-8) for t in threshs]))
    flat_p, flat_l = np.concatenate(all_preds), np.concatenate(all_labels)
    best_ods = max([(2*np.sum((flat_p>=t)*flat_l)/(2*np.sum((flat_p>=t)*flat_l)+np.sum((flat_p>=t)*(1-flat_l))+np.sum((flat_p<t)*flat_l)+1e-8), t) for t in threshs])
    return {'ODS': best_ods[0], 'ODS_thresh': best_ods[1], 'OIS': np.mean(ois), 'AP': average_precision_score(flat_l, flat_p) if np.sum(flat_l)>0 else 0}

test_loader = DataLoader(EdgeDataset(DATASET_ROOT, 'test'), batch_size=1, shuffle=False)
predictions, ground_truths, names = [], [], []
with torch.no_grad():
    for imgs, gts, ns in tqdm(test_loader):
        fuse, _ = model(imgs.to(DEVICE))
        predictions.extend([fuse[i,0].cpu().numpy() for i in range(fuse.shape[0])])
        ground_truths.extend([gts[i].cpu().numpy() for i in range(gts.shape[0])])
        names.extend(ns)

metrics = compute_metrics(predictions, ground_truths)
print(f"\n{'='*60}\nMI-Net: ODS={metrics['ODS']:.4f} | OIS={metrics['OIS']:.4f} | AP={metrics['AP']:.4f}\n{'='*60}")

import json
with open(OUTPUT_DIR / 'minet_metrics.json', 'w') as f: 
    json.dump({'model': 'MI-Net', 'bio': 'Multi-scale integration', 'improvement': 'Resolution robustness', 'metrics': metrics}, f, indent=2)
print(f"✅ MI-Net complete!")